# CivicLens: 04. Code-Mixed Linguistic Robustness Evaluation
This notebook evaluates model performance degradation when complaints shift from standard English to Romanized Indian code-mixed dialects (Hinglish/Tanglish).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from src.classification import TfidfBaselineClassifier, SemanticEmbeddingClassifier
from src.evaluation import calculate_code_mixed_robustness
from src.visualization import plot_robustness_comparison

## 1. Load Parallel Code-Mixed Dataset

In [ ]:
eval_df = pd.read_csv('../data/code_mixed_eval.csv')
print(f'Total parallel evaluation cases: {len(eval_df)}')
eval_df.head()

## 2. Train Models on Clean Complaints Corpus

In [ ]:
train_df = pd.read_csv('../data/sample_complaints.csv')
tfidf_lr = TfidfBaselineClassifier(classifier_type='logistic_regression').fit(train_df['text'].tolist(), train_df['category'].tolist())
tfidf_svm = TfidfBaselineClassifier(classifier_type='linear_svm', analyzer='char_wb', ngram_range=(3, 5)).fit(train_df['text'].tolist(), train_df['category'].tolist())
semantic_clf = SemanticEmbeddingClassifier().fit(train_df['text'].tolist(), train_df['category'].tolist())

## 3. Measure Robustness Drop (Delta F1)

In [ ]:
y_true = eval_df['category'].tolist()
std_texts = eval_df['standard_english'].tolist()
cm_texts = eval_df['code_mixed_variant'].tolist()

rob_lr = calculate_code_mixed_robustness(y_true, tfidf_lr.predict(std_texts), tfidf_lr.predict(cm_texts))
rob_svm = calculate_code_mixed_robustness(y_true, tfidf_svm.predict(std_texts), tfidf_svm.predict(cm_texts))
rob_sem = calculate_code_mixed_robustness(y_true, semantic_clf.predict(std_texts), semantic_clf.predict(cm_texts))

rob_table = pd.DataFrame([
    {'Model': 'TF-IDF + LR', 'English_F1': rob_lr['standard_macro_f1'], 'CodeMixed_F1': rob_lr['codemix_macro_f1'], 'Delta_F1': rob_lr['delta_macro_f1']},
    {'Model': 'TF-IDF Char + SVM', 'English_F1': rob_svm['standard_macro_f1'], 'CodeMixed_F1': rob_svm['codemix_macro_f1'], 'Delta_F1': rob_svm['delta_macro_f1']},
    {'Model': 'Multilingual MiniLM', 'English_F1': rob_sem['standard_macro_f1'], 'CodeMixed_F1': rob_sem['codemix_macro_f1'], 'Delta_F1': rob_sem['delta_macro_f1']}
])
rob_table

## 4. Linguistic Degradation Plot

In [ ]:
fig = plot_robustness_comparison(rob_table.to_dict('records'))
plt.show()